# HybridSiam-CD+ Combined Training
### Trains on all 4 datasets: GBCD-Binary + FOTBCD-Binary + LEVIR-CD+ + WHU-CD

**Run cells in order — each cell has a clear purpose.**

## Cell 1 — Mount Drive + Install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install albumentations timm -q
print('Packages ready!')

Mounted at /content/drive
Packages ready!


## Cell 2 — Copy files from Drive to Colab

In [ ]:
import os, shutil

# ── Copy ONLY the small files (model.py and model.pth) ───────────────────────
print('Copying model files (small, fast) ...')
shutil.copy('/content/drive/MyDrive/GBCD/model.py',  '/content/model.py')
shutil.copy('/content/drive/MyDrive/GBCD/model.pth', '/content/model.pth')
print('  model.py and model.pth copied.')

# ── Use datasets DIRECTLY from Drive — no copying needed ─────────────────────
print('\nUsing datasets directly from Google Drive (no copy needed):')
datasets = {
    'GBCD-Binary'  : '/content/drive/MyDrive/GBCD/GBCD-Binary',
    'FOTBCD-Binary': '/content/drive/MyDrive/GBCD/FOTBCD-Binary',
    'LEVIR-CD+'    : '/content/drive/MyDrive/GBCD/LEVIR-CD+',
    'whu-cd'       : '/content/drive/MyDrive/GBCD/whu-cd',
}
for name, path in datasets.items():
    exists = os.path.exists(path)
    print(f'  {"OK" if exists else "MISSING"} → {name}: {path}')

print('\nAll ready — no copying needed!')

Copying model files (small, fast) ...
  model.py and model.pth copied.

Using datasets directly from Google Drive (no copy needed):
  OK → GBCD-Binary: /content/drive/MyDrive/GBCD/GBCD-Binary
  OK → FOTBCD-Binary: /content/drive/MyDrive/GBCD/FOTBCD-Binary
  OK → LEVIR-CD+: /content/drive/MyDrive/GBCD/LEVIR-CD+
  OK → whu-cd: /content/drive/MyDrive/GBCD/whu-cd

All ready — no copying needed!


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
import zipfile, os, time

zips = ['GBCD-Binary', 'FOTBCD-Binary', 'LEVIR-CD+', 'whu-cd']

for name in zips:
    local_path = f'/content/{name}'
    zip_path   = f'/content/drive/MyDrive/GBCD/{name}.zip'

    if os.path.exists(local_path):
        print(f'  {name} already exists — skipping.')
        continue
    if not os.path.exists(zip_path):
        print(f'  ERROR: {name}.zip not found in Drive!')
        continue

    print(f'Unzipping {name} ...')
    t0 = time.time()
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/content/')
    n = sum(len(files) for _, _, files in os.walk(local_path))
    print(f'  Done in {time.time()-t0:.0f}s  ({n:,} files)')

print('\nAll datasets ready! Now run Cell 4 onwards.')

Unzipping GBCD-Binary ...
  Done in 22s  (5,307 files)
Unzipping FOTBCD-Binary ...
  Done in 505s  (83,614 files)
Unzipping LEVIR-CD+ ...
  Done in 48s  (2,955 files)
Unzipping whu-cd ...
  Done in 30s  (22,860 files)

All datasets ready! Now run Cell 4 onwards.


## Cell 3 — Imports

In [ ]:
import os, random, time, types
from pathlib import Path

import numpy as np
import cv2
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler
import albumentations as A
from albumentations.pytorch import ToTensorV2

from model import HybridChangeDetector

print('All imports OK')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

All imports OK
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB


## Cell 4 — Configuration

In [ ]:
# ── PATHS ─────────────────────────────────────────────────────────────────────
PRETRAINED_WEIGHTS = '/content/model.pth'
SAVE_DIR           = '/content/runs/combined'

DATASET_CONFIGS = [
    {
        'name'       : 'GBCD-Binary',
        'base'       : '/content/GBCD-Binary',
        'before'     : 'before',
        'after'      : 'after',
        'label'      : 'label',
        'train_split': 'train',
        'val_split'  : 'val',
        'test_split' : 'test',
    },
    {
        'name'       : 'FOTBCD-Binary',
        'base'       : '/content/FOTBCD-Binary',
        'before'     : 'before',
        'after'      : 'after',
        'label'      : 'label',
        'train_split': 'train',
        'val_split'  : 'val',
        'test_split' : 'test',
    },
    {
        'name'       : 'LEVIR-CD+',
        'base'       : '/content/LEVIR-CD+',
        'before'     : 'A',
        'after'      : 'B',
        'label'      : 'label',
        'train_split': 'train',
        'val_split'  : None,
        'test_split' : 'test',
    },
    {
        'name'       : 'whu-cd',
        'base'       : '/content/whu-cd',
        'before'     : 'A',
        'after'      : 'B',
        'label'      : 'OUT',
        'train_split': 'train',
        'val_split'  : 'val',
        'test_split' : 'test',
    },
]

# ── HYPER-PARAMETERS ──────────────────────────────────────────────────────────
IMG_SIZE        = 512
BATCH_SIZE      = 64    # doubled — 95GB GPU can handle it
NUM_EPOCHS      = 40
FREEZE_VIT      = True
LR_DECODER      = 1e-5
LR_CBAM         = 1e-4
LR_FUSION       = 1e-4
WEIGHT_DECAY    = 1e-4
WARMUP_EPOCHS   = 3
PATIENCE        = 8
FOCAL_GAMMA     = 2.5
FOCAL_ALPHA     = 0.75
GRAD_CLIP       = 1.0
VAL_SPLIT_RATIO = 0.10
NUM_WORKERS     = 8     # increased — faster data loading
SEED            = 42

MEAN       = [0.485, 0.456, 0.406]
STD        = [0.229, 0.224, 0.225]
EXTENSIONS = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')

import os
os.makedirs(SAVE_DIR, exist_ok=True)
print('Configuration set.')
print(f'IMG_SIZE={IMG_SIZE}  BATCH_SIZE={BATCH_SIZE}  NUM_WORKERS={NUM_WORKERS}')
print(f'SAVE_DIR: {SAVE_DIR}')

Configuration set.
IMG_SIZE=512  BATCH_SIZE=64  NUM_WORKERS=8
SAVE_DIR: /content/runs/combined


## Cell 5 — CBAM + Model definition

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(
            self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size,
                                 padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx  = x.max(dim=1, keepdim=True)[0]
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(spatial_kernel)
    def forward(self, x):
        return self.sa(self.ca(x))

class LearnableTemporalFusion(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Linear(in_dim * 2, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
        )
    def forward(self, feat_before, feat_after):
        return self.conv(torch.cat([feat_before, feat_after], dim=-1))

class HybridSiamPlus(nn.Module):
    def __init__(self, pretrained_path):
        super().__init__()
        self.base            = HybridChangeDetector(pretrained=False)
        self._load_pretrained(pretrained_path)
        if FREEZE_VIT:
            for p in self.base.encoder.backbone.parameters():
                p.requires_grad = False
            print('[VRAM] ViT backbone frozen.')
        self.cbam_bottleneck = CBAM(256)
        self.cbam_fuse1      = CBAM(128)
        self.cbam_fuse2      = CBAM(64)
        self.cbam_fuse3      = CBAM(64)
        self.cbam_fuse4      = CBAM(32)
        enc_dim = self.base.encoder.out_dim
        self.temporal_fusion = LearnableTemporalFusion(enc_dim, enc_dim)
        self._patch_encoder()
        self._patch_decoder()

    def _load_pretrained(self, path):
        state = torch.load(path, map_location='cpu', weights_only=False)
        for key in ('model', 'state_dict'):
            if isinstance(state, dict) and key in state:
                state = state[key]
        base_state  = self.base.state_dict()
        compatible  = {k: v for k, v in state.items()
                       if k in base_state and
                       base_state[k].shape == v.shape}
        self.base.load_state_dict(compatible, strict=False)
        print(f'[Weights] {len(compatible)}/{len(base_state)} keys loaded.')

    def _patch_encoder(self):
        fusion = self.temporal_fusion
        def _forward(self_enc, before, after):
            feat_before = self_enc.backbone(before)
            feat_after  = self_enc.backbone(after)
            num_prefix  = getattr(self_enc.backbone, 'num_prefix_tokens', 1)
            feat_before = feat_before[:, num_prefix:]
            feat_after  = feat_after[:, num_prefix:]
            feat_before = self_enc.proj(feat_before)
            feat_after  = self_enc.proj(feat_after)
            return fusion(feat_before, feat_after)
        self.base.encoder.forward = types.MethodType(
            _forward, self.base.encoder)

    def _patch_decoder(self):
        decoder = self.base.decoder
        cbam_bn = self.cbam_bottleneck
        cbam_f1 = self.cbam_fuse1
        cbam_f2 = self.cbam_fuse2
        cbam_f3 = self.cbam_fuse3
        cbam_f4 = self.cbam_fuse4
        def _forward(self_dec, vit_features, resnet_features, output_size):
            B, N, D = vit_features.shape
            h = w = int(N ** 0.5)
            vit = vit_features.permute(0, 2, 1).view(B, D, h, w)
            s0, s1, s2, s3, s4, s5 = resnet_features
            s5_up = F.interpolate(s5, size=(h, w), mode='nearest')
            x = self_dec.bottleneck(torch.cat([vit, s5_up, s4], dim=1))
            x = cbam_bn(x)
            x = self_dec.up1(x)
            x = self_dec.fuse1(torch.cat([x, s3], dim=1))
            x = cbam_f1(x)
            x = self_dec.up2(x)
            x = self_dec.fuse2(torch.cat([x, s2], dim=1))
            x = cbam_f2(x)
            x = self_dec.up3(x)
            x = self_dec.fuse3(torch.cat([x, s1], dim=1))
            x = cbam_f3(x)
            x = self_dec.up4(x)
            x = self_dec.fuse4(torch.cat([x, s0], dim=1))
            x = cbam_f4(x)
            x = self_dec.head(x)
            if x.shape[2:] != output_size:
                x = F.interpolate(x, size=output_size, mode='nearest')
            return x
        decoder.forward = types.MethodType(_forward, decoder)

    def forward(self, before, after):
        return self.base(before, after)

print('Model class defined.')

Model class defined.


## Cell 6 — Loss + Dataset + Metrics

In [ ]:
# ── Make sure these are all in Cell 6 ────────────────────────────────────────

# LOSS
class FocalDiceLoss(nn.Module):
    def __init__(self, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, smooth=1.0):
        super().__init__()
        self.alpha  = alpha
        self.gamma  = gamma
        self.smooth = smooth

    def _focal(self, logits, target):
        pred    = logits.squeeze(1)
        target  = target.float()
        bce     = F.binary_cross_entropy_with_logits(
                      pred, target, reduction='none')
        p_t     = torch.sigmoid(pred)*target + \
                  (1-torch.sigmoid(pred))*(1-target)
        alpha_t = self.alpha*target + (1-self.alpha)*(1-target)
        return (alpha_t * (1-p_t)**self.gamma * bce).mean()

    def _dice(self, logits, target):
        pred   = torch.sigmoid(logits)
        if target.dim() == 3:
            target = target.unsqueeze(1)
        target = target.float()
        p_flat = pred.view(pred.size(0), -1)
        t_flat = target.view(target.size(0), -1)
        inter  = (p_flat * t_flat).sum(dim=1)
        denom  = p_flat.sum(dim=1) + t_flat.sum(dim=1)
        return (1-(2*inter+self.smooth)/(denom+self.smooth)).mean()

    def forward(self, logits, target):
        f = self._focal(logits, target)
        d = self._dice(logits, target)
        if torch.isnan(f) or torch.isnan(d):
            return torch.tensor(0.0, requires_grad=True,
                                device=logits.device)
        return f + d

# FIND LABEL HELPER  ← this was missing!
def find_label(label_dir, stem):
    for ext in EXTENSIONS:
        p = os.path.join(label_dir, stem + ext)
        if os.path.exists(p): return p
    return None

# DATASET
class ChangeDetectionDataset(Dataset):
    def __init__(self, base_path, split_name,
                 before_sub, after_sub, label_sub,
                 img_size=IMG_SIZE, augment=True,
                 sample_indices=None, dataset_name=''):
        super().__init__()
        self.img_size     = img_size
        self.dataset_name = dataset_name
        split_dir  = os.path.join(base_path, split_name)
        before_dir = os.path.join(split_dir, before_sub)
        after_dir  = os.path.join(split_dir, after_sub)
        label_dir  = os.path.join(split_dir, label_sub)

        for d in (before_dir, after_dir, label_dir):
            if not os.path.exists(d):
                raise FileNotFoundError(f'Not found: {d}')

        all_samples = []
        for name in sorted(os.listdir(before_dir)):
            if not name.lower().endswith(EXTENSIONS): continue
            stem  = Path(name).stem
            a_p   = os.path.join(after_dir, name)
            l_p   = find_label(label_dir, stem)
            if os.path.exists(a_p) and l_p:
                all_samples.append(
                    (os.path.join(before_dir, name), a_p, l_p))

        if sample_indices is not None:
            self.samples = [all_samples[i] for i in sample_indices]
        else:
            self.samples = all_samples

        if not self.samples:
            raise RuntimeError(f'No samples in {split_dir}')

        if augment:
            self.tf = A.Compose([
                A.Resize(img_size, img_size),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.Transpose(p=0.3),
                A.ColorJitter(brightness=0.2, contrast=0.2,
                              saturation=0.1, hue=0.05, p=0.5),
                A.GaussNoise(p=0.2),
                A.Normalize(mean=MEAN, std=STD),
                ToTensorV2(),
            ], additional_targets={'image2': 'image'},
               is_check_shapes=False)
        else:
            self.tf = A.Compose([
                A.Resize(img_size, img_size),
                A.Normalize(mean=MEAN, std=STD),
                ToTensorV2(),
            ], additional_targets={'image2': 'image'},
               is_check_shapes=False)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        b_p, a_p, l_p = self.samples[idx]
        before = cv2.cvtColor(cv2.imread(b_p, cv2.IMREAD_COLOR),
                              cv2.COLOR_BGR2RGB)
        after  = cv2.cvtColor(cv2.imread(a_p, cv2.IMREAD_COLOR),
                              cv2.COLOR_BGR2RGB)
        label  = cv2.imread(l_p, cv2.IMREAD_GRAYSCALE)
        before = cv2.resize(before, (self.img_size, self.img_size))
        after  = cv2.resize(after,  (self.img_size, self.img_size))
        label  = cv2.resize(label,  (self.img_size, self.img_size),
                            interpolation=cv2.INTER_NEAREST)
        mask   = (label > 127).astype(np.uint8)
        aug    = self.tf(image=before, image2=after, mask=mask)
        return {
            'A':    aug['image'].float(),
            'B':    aug['image2'].float(),
            'mask': aug['mask'].float(),
        }

# METRICS
class Metrics:
    def __init__(self):
        self.tp = self.fp = self.fn = self.tn = 0.0

    @torch.no_grad()
    def update(self, logits, target):
        pred   = (torch.sigmoid(logits.squeeze(1)) > 0.5).float()
        target = target.float()
        self.tp += (pred * target).sum().item()
        self.fp += (pred * (1-target)).sum().item()
        self.fn += ((1-pred) * target).sum().item()
        self.tn += ((1-pred) * (1-target)).sum().item()

    def compute(self):
        tp, fp, fn, tn = self.tp, self.fp, self.fn, self.tn
        precision = tp / (tp+fp+1e-8)
        recall    = tp / (tp+fn+1e-8)
        f1        = 2*precision*recall / (precision+recall+1e-8)
        iou       = tp / (tp+fp+fn+1e-8)
        accuracy  = (tp+tn) / (tp+fp+fn+tn+1e-8)
        return dict(precision=precision, recall=recall,
                    f1=f1, iou=iou, accuracy=accuracy)

print('Loss, find_label, Dataset, Metrics all defined.')

Loss, find_label, Dataset, Metrics all defined.


## Cell 7 — Build combined train and val datasets

In [ ]:
random.seed(SEED)
np.random.seed(SEED)

train_datasets = []
val_datasets   = []

for cfg in DATASET_CONFIGS:
    name = cfg['name']
    print(f'\nBuilding {name} ...')

    # ── Training set ──────────────────────────────────────────────────────────
    if cfg['val_split'] is None:
        # LEVIR-CD+: no val — split 10% from train
        full_ds = ChangeDetectionDataset(
            cfg['base'], cfg['train_split'],
            cfg['before'], cfg['after'], cfg['label'],
            augment=False, dataset_name=name)
        n       = len(full_ds.samples)
        indices = list(range(n))
        random.shuffle(indices)
        n_val   = max(1, int(n * VAL_SPLIT_RATIO))
        val_idx = indices[:n_val]
        trn_idx = indices[n_val:]

        train_ds = ChangeDetectionDataset(
            cfg['base'], cfg['train_split'],
            cfg['before'], cfg['after'], cfg['label'],
            augment=True, sample_indices=trn_idx,
            dataset_name=name)
        val_ds = ChangeDetectionDataset(
            cfg['base'], cfg['train_split'],
            cfg['before'], cfg['after'], cfg['label'],
            augment=False, sample_indices=val_idx,
            dataset_name=name)
        print(f'  Auto-split: train={len(train_ds)} val={len(val_ds)}')
    else:
        train_ds = ChangeDetectionDataset(
            cfg['base'], cfg['train_split'],
            cfg['before'], cfg['after'], cfg['label'],
            augment=True, dataset_name=name)
        val_ds = ChangeDetectionDataset(
            cfg['base'], cfg['val_split'],
            cfg['before'], cfg['after'], cfg['label'],
            augment=False, dataset_name=name)
        print(f'  train={len(train_ds)} val={len(val_ds)}')

    train_datasets.append(train_ds)
    val_datasets.append(val_ds)

# ── Combine all datasets ──────────────────────────────────────────────────────
combined_train = ConcatDataset(train_datasets)
combined_val   = ConcatDataset(val_datasets)

print(f'\n{"="*55}')
print(f'Combined train : {len(combined_train):,} tiles')
print(f'Combined val   : {len(combined_val):,} tiles')

# Per-dataset counts
print('\nBreakdown:')
for cfg, td, vd in zip(DATASET_CONFIGS, train_datasets, val_datasets):
    print(f'  {cfg["name"]:<20} train={len(td):>5}  val={len(vd):>5}')


Building GBCD-Binary ...
  train=1417 val=176

Building FOTBCD-Binary ...
  train=25908 val=984

Building LEVIR-CD+ ...
  Auto-split: train=574 val=63

Building whu-cd ...
  train=6096 val=762

Combined train : 33,995 tiles
Combined val   : 1,985 tiles

Breakdown:
  GBCD-Binary          train= 1417  val=  176
  FOTBCD-Binary        train=25908  val=  984
  LEVIR-CD+            train=  574  val=   63
  whu-cd               train= 6096  val=  762


## Cell 8 — Build DataLoaders

In [ ]:
train_loader = DataLoader(
    combined_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True,
)

val_loader = DataLoader(
    combined_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

print(f'Train loader: {len(train_loader)} batches')
print(f'Val   loader: {len(val_loader)} batches')
print(f'Batch size  : {BATCH_SIZE}')
print(f'Each epoch  : ~{len(train_loader)*BATCH_SIZE:,} images')

Train loader: 531 batches
Val   loader: 32 batches
Batch size  : 64
Each epoch  : ~33,984 images


In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

free_gb  = torch.cuda.mem_get_info()[0] / 1024**3
total_gb = torch.cuda.mem_get_info()[1] / 1024**3
print(f'GPU memory: {free_gb:.1f} GB free / {total_gb:.1f} GB total')

# Auto-suggest batch size
if free_gb > 60:
    print('Suggestion: BATCH_SIZE = 48')
elif free_gb > 40:
    print('Suggestion: BATCH_SIZE = 32')
else:
    print('Suggestion: BATCH_SIZE = 24')

GPU memory: 92.2 GB free / 95.0 GB total
Suggestion: BATCH_SIZE = 48


## Cell 9 — Build model + optimizer + scheduler

In [ ]:
device = torch.device('cuda')

print('Building HybridSiam-CD+ ...')
model = HybridSiamPlus(pretrained_path=PRETRAINED_WEIGHTS)
model = model.to(device)

# Parameter groups
vit_ids    = set(id(p) for p in model.base.encoder.backbone.parameters())
cbam_ids   = (
    set(id(p) for p in model.cbam_bottleneck.parameters()) |
    set(id(p) for p in model.cbam_fuse1.parameters())      |
    set(id(p) for p in model.cbam_fuse2.parameters())      |
    set(id(p) for p in model.cbam_fuse3.parameters())      |
    set(id(p) for p in model.cbam_fuse4.parameters())
)
fusion_ids = set(id(p) for p in model.temporal_fusion.parameters())

cbam_params   = [p for p in model.parameters()
                 if id(p) in cbam_ids and p.requires_grad]
fusion_params = [p for p in model.parameters()
                 if id(p) in fusion_ids and p.requires_grad]
other_params  = [p for p in model.parameters()
                 if id(p) not in vit_ids
                 and id(p) not in cbam_ids
                 and id(p) not in fusion_ids
                 and p.requires_grad]

optimizer = optim.AdamW([
    {'params': other_params,  'lr': LR_DECODER},
    {'params': cbam_params,   'lr': LR_CBAM},
    {'params': fusion_params, 'lr': LR_FUSION},
], weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch+1) / WARMUP_EPOCHS
    progress = (epoch-WARMUP_EPOCHS) / max(NUM_EPOCHS-WARMUP_EPOCHS, 1)
    return 0.01 + 0.99*0.5*(1+np.cos(np.pi*progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
criterion = FocalDiceLoss()
scaler    = torch.cuda.amp.GradScaler()   # AMP — safe on 80GB GPU

n_total     = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters : {n_total:,} total | {n_trainable:,} trainable | '
      f'{n_total-n_trainable:,} frozen')
print(f'Decoder    : {sum(p.numel() for p in other_params):,}  LR={LR_DECODER}')
print(f'CBAM       : {sum(p.numel() for p in cbam_params):,}  LR={LR_CBAM}')
print(f'Fusion     : {sum(p.numel() for p in fusion_params):,}  LR={LR_FUSION}')

Building HybridSiam-CD+ ...
[Weights] 687/687 keys loaded.
[VRAM] ViT backbone frozen.
Parameters : 329,434,637 total | 26,355,213 trainable | 303,079,424 frozen
Decoder    : 26,211,363  LR=1e-05
CBAM       : 12,010  LR=0.0001
Fusion     : 131,840  LR=0.0001


/tmp/ipykernel_992/3893952281.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()   # AMP — safe on 80GB GPU


## Cell 10 — Validation function

In [ ]:
@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    metrics    = Metrics()
    total_loss = 0.0
    counted    = 0
    for batch in loader:
        before = batch['A'].to(device)
        after  = batch['B'].to(device)
        mask   = batch['mask'].to(device)
        if mask.sum() == 0: continue
        with torch.cuda.amp.autocast():
            pred = model(before, after)
            loss = criterion(pred, mask)
        if not torch.isnan(loss):
            total_loss += loss.item()
            counted += 1
        metrics.update(pred, mask)
    m         = metrics.compute()
    m['loss']  = total_loss / max(counted, 1)
    model.train()
    return m

print('Validation function ready.')

Validation function ready.


In [ ]:
import shutil

# Restore best model from Drive to resume training
drive_best = '/content/drive/MyDrive/GBCD/best_model_combined.pth'
if os.path.exists(drive_best):
    shutil.copy(drive_best,
                os.path.join(SAVE_DIR, 'best_model.pth'))
    print(f'Restored best model from Drive!')
    print(f'Best IoU so far: 0.8033 (epoch 30)')
else:
    print('No saved model found in Drive — starting fresh.')

No saved model found in Drive — starting fresh.


## Cell 11 — TRAINING LOOP

In [ ]:
log_path    = os.path.join(SAVE_DIR, 'training_log.csv')
resume_path = os.path.join(SAVE_DIR, 'latest_checkpoint.pth')

# Resume if checkpoint exists
start_epoch = 0
best_iou    = 0.0
no_improve  = 0

if os.path.exists(resume_path):
    print(f'Resuming from checkpoint ...')
    ckpt = torch.load(resume_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    start_epoch = ckpt['epoch'] + 1
    best_iou    = ckpt['best_iou']
    no_improve  = ckpt['no_improve']
    print(f'  Resumed at epoch {start_epoch}, best_iou={best_iou:.4f}')
else:
    with open(log_path, 'w') as f:
        f.write('epoch,train_loss,train_iou,train_f1,'
                'val_loss,val_iou,val_f1,'
                'val_precision,val_recall,lr\n')

print(f'\n{"="*65}')
print(f'  HybridSiam-CD+ — Combined Training')
print(f'  Datasets: GBCD + FOTBCD + LEVIR + WHU')
print(f'  Train tiles : {len(combined_train):,}')
print(f'  Val tiles   : {len(combined_val):,}')
print(f'  Epochs      : {NUM_EPOCHS}  patience={PATIENCE}')
print(f'  Batch size  : {BATCH_SIZE}  AMP: ON')
print(f'{"="*65}\n')

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    train_m     = Metrics()
    train_loss  = 0.0
    nan_batches = 0
    t0          = time.time()

    pbar = tqdm(train_loader,
                desc=f'Epoch {epoch+1:03d}/{NUM_EPOCHS}',
                leave=False)

    for batch in pbar:
        before = batch['A'].to(device, non_blocking=True)
        after  = batch['B'].to(device, non_blocking=True)
        mask   = batch['mask'].to(device, non_blocking=True)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model(before, after)
            loss = criterion(pred, mask)

        if torch.isnan(loss) or torch.isinf(loss):
            nan_batches += 1
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        train_m.update(pred.detach(), mask)
        train_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    scheduler.step()

    tm     = train_m.compute()
    t_loss = train_loss / max(len(train_loader)-nan_batches, 1)
    vm     = validate(model, val_loader, criterion, device)
    lr_now = scheduler.get_last_lr()[0]
    secs   = time.time() - t0

    nan_msg = f' [{nan_batches} nan]' if nan_batches else ''
    print(
        f'Epoch {epoch+1:03d}/{NUM_EPOCHS}  '
        f'Train loss={t_loss:.4f} IoU={tm["iou"]:.4f} '
        f'F1={tm["f1"]:.4f}  |  '
        f'Val loss={vm["loss"]:.4f} IoU={vm["iou"]:.4f} '
        f'F1={vm["f1"]:.4f} P={vm["precision"]:.4f} '
        f'R={vm["recall"]:.4f}  '
        f'LR={lr_now:.2e}  [{secs:.0f}s]{nan_msg}'
    )

    with open(log_path, 'a') as f:
        f.write(f'{epoch+1},{t_loss:.6f},{tm["iou"]:.6f},'
                f'{tm["f1"]:.6f},{vm["loss"]:.6f},{vm["iou"]:.6f},'
                f'{vm["f1"]:.6f},{vm["precision"]:.6f},'
                f'{vm["recall"]:.6f},{lr_now:.2e}\n')

    if vm['iou'] > best_iou:
        best_iou   = vm['iou']
        no_improve = 0
        torch.save(model.state_dict(),
                   os.path.join(SAVE_DIR, 'best_model.pth'))
        print(f'  ✓ New best IoU: {best_iou:.4f} → saved best_model.pth')
    else:
        no_improve += 1
        print(f'  No improvement ({no_improve}/{PATIENCE})')

    torch.save({
        'epoch':      epoch,
        'model':      model.state_dict(),
        'optimizer':  optimizer.state_dict(),
        'scheduler':  scheduler.state_dict(),
        'scaler':     scaler.state_dict(),
        'best_iou':   best_iou,
        'no_improve': no_improve,
    }, resume_path)

    if no_improve >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1}.')
        break

# Save locally
        local_best = os.path.join(SAVE_DIR, 'best_model.pth')
        torch.save(model.state_dict(), local_best)
        # Save to Drive immediately — never lose progress
        drive_best = '/content/drive/MyDrive/GBCD/best_model_combined.pth'
        shutil.copy(local_best, drive_best)
        print(f'  ✓ New best IoU: {best_iou:.4f} → saved locally + Drive')

print(f'\n{"="*65}')
print(f'Training complete.  Best val IoU : {best_iou:.4f}')
print(f'Best model  → {SAVE_DIR}/best_model.pth')
print(f'{"="*65}')


  HybridSiam-CD+ — Combined Training
  Datasets: GBCD + FOTBCD + LEVIR + WHU
  Train tiles : 33,995
  Val tiles   : 1,985
  Epochs      : 40  patience=8
  Batch size  : 64  AMP: ON



Epoch 001/40:   0%|          | 0/531 [00:00<?, ?it/s]

/tmp/ipykernel_992/2277454642.py:52: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_992/169090822.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 001/40  Train loss=0.5005 IoU=0.2688 F1=0.4238  |  Val loss=0.3520 IoU=0.6010 F1=0.7508 P=0.7248 R=0.7786  LR=6.67e-06  [852s]
  ✓ New best IoU: 0.6010 → saved best_model.pth


Epoch 002/40:   0%|          | 0/531 [00:00<?, ?it/s]

...

Epoch 038/40  Train loss=0.0782 IoU=0.8502 F1=0.9190  |  Val loss=0.1716 IoU=0.8005 F1=0.8892 P=0.8902 R=0.8882  LR=1.71e-07  [851s]
  No improvement (6/8)


Epoch 039/40:   0%|          | 0/531 [00:00<?, ?it/s]

Epoch 039/40  Train loss=0.0785 IoU=0.8496 F1=0.9187  |  Val loss=0.1718 IoU=0.7992 F1=0.8884 P=0.8915 R=0.8853  LR=1.18e-07  [851s]
  No improvement (7/8)


Epoch 040/40:   0%|          | 0/531 [00:00<?, ?it/s]

Epoch 040/40  Train loss=0.0795 IoU=0.8458 F1=0.9164  |  Val loss=0.1694 IoU=0.8000 F1=0.8889 P=0.8902 R=0.8876  LR=1.00e-07  [851s]
  No improvement (8/8)

Early stopping at epoch 40.

Training complete.  Best val IoU : 0.8015
Best model  → /content/runs/combined/best_model.pth


## Cell 12 — Save best model to Google Drive

In [ ]:
import shutil
drive_save = '/content/drive/MyDrive/GBCD/best_model_combined.pth'
shutil.copy(os.path.join(SAVE_DIR, 'best_model.pth'), drive_save)
shutil.copy(log_path,
            '/content/drive/MyDrive/GBCD/combined_training_log.csv')
print(f'Best model saved to Drive → {drive_save}')
print(f'Training log saved to Drive.')

Best model saved to Drive → /content/drive/MyDrive/GBCD/best_model_combined.pth
Training log saved to Drive.
